In [1]:
# 4. Variant list and lexical retrieval

import re
from collections import Counter
from pathlib import Path

from gensim.models import FastText
import pandas as pd

DATA = Path("../data")   # not included in the repository
TAB = Path("../tab")

In [2]:
# Definitions

NIGHT = ["nacht", "avondschemering", "ochtendschemering"]   # night incl. evening/morning twilight


# Seed forms of the variant list: five noise terms, their "nacht-" compounds, and "nachtrust".
SEEDS = ["lawijd", "lawaai", "gerucht", "kabaal", "rumoer",
         "nachtlawijd", "nachtlawaai", "nachtrumoer", "nachtgerucht", "nachtrust"]


def event_id(df):
    """Unique incident key: register + incident number."""
    number = pd.to_numeric(df["Incident Number"], errors="coerce").astype("Int64").astype(str)
    return df["archive_file"].astype(str).str.strip() + "_" + number


def load_incidents(folder, columns=("archive_file", "Incident Number", "year", "district", "astro", "incident_text"),
                   one_row=True):
    """All incidents of all districts. The tables have one row per person involved; with one_row=True
    only the first row of each incident is kept."""
    files = [f for f in sorted(folder.glob("*.xlsx")) if not f.name.startswith(("~$", "._"))]
    df = pd.concat([pd.read_excel(f, usecols=list(columns)) for f in files], ignore_index=True)
    df["event_id"] = event_id(df)
    return (df.drop_duplicates("event_id") if one_row else df).reset_index(drop=True)


def word_set(text):
    """Words of an incident description as used for lexical retrieval."""
    if not isinstance(text, str):
        return set()
    return {re.sub(r"[^\w]", "", w) for w in text.lower().split()}


def variant_groups(fasttext_model, threshold=0.75, topn=300):
    """FastText neighbours of each seed with cosine >= threshold. Neighbours with a
    "nacht-" prefix are only kept for seeds that carry that prefix themselves, and vice versa."""
    groups = {}
    for seed in SEEDS:
        forms = {seed}
        for word, score in fasttext_model.wv.most_similar(seed, topn=topn):
            if score >= threshold and word.startswith("nacht") == seed.startswith("nacht"):
                forms.add(word)
        groups[seed] = forms
    return groups

In [3]:
# Variant list

ft_rare = FastText.load(str(DATA / "models" / "FastText_htr.model"))
groups = variant_groups(ft_rare, threshold=0.75)
forms = set().union(*groups.values())

pd.DataFrame([(seed, form) for seed, fs in groups.items() for form in sorted(fs)],
             columns=["seed", "form"]).to_csv(TAB / "variant_list.csv", index=False)
print(f"{len(forms)} unique forms in {len(groups)} groups")
print({seed: len(fs) for seed, fs in groups.items()})

695 unique forms in 10 groups
{'lawijd': 197, 'lawaai': 236, 'gerucht': 69, 'kabaal': 40, 'rumoer': 90, 'nachtlawijd': 170, 'nachtlawaai': 203, 'nachtrumoer': 265, 'nachtgerucht': 271, 'nachtrust': 117}


In [4]:
# Retrieval

rows = load_incidents(DATA / "incidents", one_row=False)          # one row per person involved
incidents = rows.drop_duplicates("event_id").copy()
incidents["matched"] = incidents["incident_text"].map(lambda t: sorted(word_set(t) & forms))
hits = incidents[incidents["matched"].map(len) > 0].copy()
hits["matched"] = hits["matched"].map("; ".join)

# An incident counts as nocturnal when the time of day of any person row is night.
hit_rows = rows[rows["event_id"].isin(hits["event_id"])]
hits["time_known"] = hits["event_id"].isin(hit_rows.loc[hit_rows["astro"].notna(), "event_id"])
hits["night"] = hits["event_id"].isin(hit_rows.loc[hit_rows["astro"].isin(NIGHT), "event_id"])
hits[["event_id", "year", "district", "night", "matched"]].to_csv(TAB / "variant_list_matches.csv", index=False)

matched_forms = Counter(f for m in hits["matched"] for f in m.split("; "))
print(f"incidents searched:            {len(incidents):,}")
print(f"retrieved, all districts:      {hits['time_known'].sum():,} (time of day known)")
print(f"  of which nocturnal:          {hits['night'].sum():,}")
print(f"retrieved, district 3, night:  {((hits['district'] == 3) & hits['night']).sum():,}")
print(f"distinct forms matched:        {len(matched_forms)}")
print("most frequent:", matched_forms.most_common(15))

incidents searched:            165,054
retrieved, all districts:      2,221 (time of day known)
  of which nocturnal:          1,722
retrieved, district 3, night:  742
distinct forms matched:        579
most frequent: [('nachtrust', 796), ('stoorde', 621), ('luidkeels', 543), ('gerucht', 263), ('kabaal', 169), ('gestoord', 138), ('lawijd', 131), ('schreeuwen', 112), ('lawijt', 83), ('lawaai', 79), ('nachtgerucht', 75), ('nachtrumoer', 66), ('nachtlawijd', 65), ('luidruchtig', 39), ('ramoer', 35)]
